# Stablecoin Risk Lab
### Un stablecoin adossé au dirham est-il viable et résilient ?

**Auteur :** Ahmed Badi — Ingénieur financier (EMSI Rabat)

## Introduction

Les **stablecoins** sont des crypto-actifs conçus pour conserver une valeur fixe
(généralement 1 dollar), grâce à des réserves détenues par leur émetteur.
Ils sont aujourd'hui au cœur des transformations de la finance : paiements
instantanés, transferts internationaux et commerce automatisé.

Au Maroc, un projet de loi sur les cryptoactifs prévoit de placer les stablecoins
sous la supervision de **Bank Al-Maghrib** et de réserver leur émission aux
**banques et établissements de paiement**.

**Problématique :**
> Un stablecoin adossé au dirham, émis par une banque marocaine, peut-il être
> à la fois rentable et résilient face aux risques de marché et de liquidité ?

**Démarche :**
1. Analyse empirique des stablecoins existants (USDT, USDC, DAI)
2. Mesure des risques : écart au peg, volatilité, VaR
3. Étude des crises historiques (Terra 2022, SVB 2023)
4. Cas d'usage : transferts des Marocains résidant à l'étranger
5. Simulation d'un émetteur fictif « MADc » : rentabilité et stress test Monte Carlo
6. Dashboard interactif et agent d'analyse IA

## 1. Préparation de l'environnement de travail

In [1]:
!pip install -q yfinance

import yfinance as yf
import pandas as pd
import numpy as np
import plotly.express as px

print("✅ Tout est prêt")

✅ Tout est prêt


**Explication du code :**
Nous installons puis importons les bibliothèques Python nécessaires au projet :
- **yfinance** : télécharge les données de marché depuis Yahoo Finance ;
- **pandas** : organise les données sous forme de tableaux (DataFrames) ;
- **numpy** : effectue les calculs numériques et statistiques ;
- **plotly** : produit des graphiques interactifs (zoom, survol).

**Résultat :** l'environnement est opérationnel, les outils sont prêts à être utilisés.

In [2]:
usdc = yf.Ticker("USDC-USD")
data = usdc.history(period="1mo")
data.head()

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2026-09-29 00:00:00+00:00,0.999929,1.000464,0.999778,0.999988,16641739658,0.0,0.0
2026-09-30 00:00:00+00:00,0.999996,1.000490,0.999533,1.000010,18071148843,0.0,0.0
2026-10-01 00:00:00+00:00,1.000007,1.000616,0.999639,0.999984,19250070080,0.0,0.0
2026-10-02 00:00:00+00:00,1.000004,1.000231,0.999439,0.999930,22153872511,0.0,0.0
2026-10-03 00:00:00+00:00,1.000015,1.000108,0.999850,0.999943,5908448581,0.0,0.0


**Explication du code :**
Nous testons la connexion aux données en téléchargeant le prix de l'**USDC**
(stablecoin émis par Circle) exprimé en dollars, sur le dernier mois.
Le ticker `USDC-USD` correspond au prix d'un USDC en dollars américains.

Le tableau obtenu contient pour chaque jour :
- **Open / Close** : prix d'ouverture et de clôture ;
- **High / Low** : prix le plus haut et le plus bas de la journée ;
- **Volume** : montant total échangé.

**Interprétation :**
Tous les prix sont très proches de **1,00 $**, avec des écarts de l'ordre de
quelques dix-millièmes. Cela confirme que l'USDC remplit sa fonction : maintenir
une valeur stable de 1 dollar. Ce lien fixe s'appelle le **peg**.

Ces petits écarts, en apparence négligeables, sont l'objet central de notre analyse :
ce sont eux qui, lors des crises, peuvent devenir brutaux et révéler la fragilité
d'un stablecoin.

## 2. Collecte des données historiques

Nous téléchargeons les prix journaliers de clôture des trois principaux stablecoins
depuis janvier 2020 :
- **USDT** (Tether) : réserves en dollars, transparence limitée ;
- **USDC** (Circle) : réserves en dollars, forte transparence ;
- **DAI** (MakerDAO) : garanti par des cryptoactifs.

Cette période couvre plusieurs crises majeures (effondrement de Terra en 2022,
faillite de Silicon Valley Bank en 2023), ce qui permettra d'analyser
le comportement des stablecoins en période de stress.

In [3]:
# Dictionnaire : nom court -> ticker Yahoo Finance
tickers = {
    "USDT": "USDT-USD",
    "USDC": "USDC-USD",
    "DAI":  "DAI-USD"
}

# Liste vide qui va recevoir les prix de chaque stablecoin
series = []

# Boucle : on répète l'opération pour chaque stablecoin
for nom, code in tickers.items():
    historique = yf.Ticker(code).history(start="2020-01-01")  # télécharger depuis 2020
    cloture = historique["Close"].rename(nom)                  # garder le prix de clôture
    series.append(cloture)                                     # l'ajouter à la liste

# Coller les 3 colonnes côte à côte
prix = pd.concat(series, axis=1)

# Retirer le fuseau horaire des dates (plus simple pour la suite)
prix.index = prix.index.tz_localize(None)

print("✅ Données téléchargées :", prix.shape)

✅ Données téléchargées : (2494, 3)


**Explication du code :**
Pour chaque stablecoin, nous téléchargeons l'historique depuis le 1er janvier 2020
et conservons uniquement le **prix de clôture** (Close), qui est la référence
standard en analyse financière. Les trois séries sont ensuite regroupées dans
un tableau unique nommé `prix`, où chaque ligne est une date et chaque colonne
un stablecoin.

**Résultat :** le tableau contient environ 2 493 jours d'observations pour
3 stablecoins, ce qui constitue une base solide pour l'analyse statistique.

In [4]:
prix.describe().round(4)

,USDT,USDC,DAI
count,2494.0000,2494.0000,2494.0000
mean,1.0002,1.0003,1.0011
std,0.0019,0.0029,0.0053
min,0.9742,0.9701,0.9648
25%,0.9997,0.9999,0.9997
50%,1.0001,1.0000,1.0000
75%,1.0004,1.0001,1.0004
max,1.0536,1.0440,1.0930


In [5]:
print("📉 Prix le plus bas de chaque stablecoin :\n")
for nom in prix.columns:
    date_min = prix[nom].idxmin()   # date du prix minimum
    valeur_min = prix[nom].min()    # valeur du prix minimum
    print(f"{nom} : {valeur_min:.4f} $ le {date_min.date()}")

📉 Prix le plus bas de chaque stablecoin :

USDT : 0.9742 $ le 2020-03-18
USDC : 0.9701 $ le 2020-03-18
DAI : 0.9648 $ le 2020-02-19


In [6]:
# Télécharger le prix le plus BAS de chaque jour pour l'USDC (colonne Low)
usdc_complet = yf.Ticker("USDC-USD").history(start="2023-03-08", end="2023-03-16")
usdc_complet.index = usdc_complet.index.tz_localize(None)

# Comparer clôture et plus bas du jour
usdc_complet[["Close", "Low"]].round(4)

,Close,Low
Date,,
2023-03-08,0.9999,0.9996
2023-03-09,1.0000,0.9994
2023-03-10,0.9995,0.9992
2023-03-11,0.9715,0.8774
2023-03-12,0.9921,0.9470
2023-03-13,0.9989,0.9880
2023-03-14,0.9992,0.9976
2023-03-15,1.0002,0.9979


**Interprétation des statistiques :**
- Sur 2 493 jours d'observation, les **moyennes** des trois stablecoins sont comprises
  entre 1,0002 $ et 1,0011 $ : sur le long terme, le peg est globalement respecté.
- Les **écarts-types** sont très faibles mais révèlent une hiérarchie :
  USDT (0,0019) < USDC (0,0029) < DAI (0,0053). Le DAI, garanti par des
  cryptoactifs volatils, est le moins stable des trois.
- Les **minimums de clôture** se situent tous début 2020 : USDT (0,9742 $) et
  USDC (0,9701 $) le 18 mars 2020, en plein krach lié au Covid-19, et DAI (0,9648 $)
  en février 2020, période où la liquidité de ces marchés était encore faible.
- Les **maximums** (jusqu'à 1,0930 $ pour le DAI) montrent que le peg peut aussi
  être rompu **à la hausse**, lorsque la demande de stablecoins explose en période de panique.

**Limite méthodologique importante :**
La crise de Silicon Valley Bank (mars 2023), durant laquelle l'USDC est tombé
autour de 0,87 $, n'apparaît pas dans les minimums. En effet, le **prix de clôture**
ne capture qu'une valeur par jour et **lisse les chocs intra-journaliers**.
Le prix le plus bas de la journée (Low) révèle un décrochage bien plus sévère.

**Conclusion intermédiaire :** la stabilité moyenne masque des risques extrêmes,
rares mais brutaux, que les données journalières tendent à sous-estimer. Ce type
de risque, dit **risque de queue**, sera mesuré dans les parties suivantes.

In [7]:
prix.to_csv("prix_stablecoins.csv")
print("💾 Fichier sauvegardé")

💾 Fichier sauvegardé


## 3. Mesure de l'écart au peg

Un stablecoin est jugé sur sa capacité à rester proche de 1 $.
Nous mesurons donc chaque jour son **écart au peg**, exprimé en **points de base**
(1 pb = 0,01 %), unité standard en finance pour les faibles variations :

> Écart (pb) = (Prix − 1) × 10 000

Un écart positif signifie que le stablecoin vaut plus de 1 $ (forte demande),
un écart négatif qu'il vaut moins de 1 $ (perte de confiance).

In [8]:
# Écart au peg en points de base
ecart = (prix - 1) * 10000

ecart.describe().round(1)

,USDT,USDC,DAI
count,2494.0,2494.0,2494.0
mean,2.3,2.9,11.3
std,18.6,28.8,52.7
min,-257.5,-298.8,-351.5
25%,-2.5,-1.4,-2.6
50%,1.2,-0.1,-0.4
75%,4.4,1.3,3.6
max,535.9,440.3,929.5


**Explication du code :**
Nous transformons les prix en écarts au peg exprimés en points de base.
Le tableau `ecart` a la même structure que `prix` : une ligne par jour,
une colonne par stablecoin. Une valeur de −20 signifie que le stablecoin
valait 0,9980 $ ce jour-là.

In [9]:
fig = px.line(prix,
              title="Prix des stablecoins depuis 2020",
              labels={"value": "Prix ($)", "Date": "Date", "variable": "Stablecoin"})

fig.add_hline(y=1, line_dash="dash", line_color="black")  # ligne du peg à 1 $
fig.show()

**Interprétation du graphique 1 :**
- **2020 : une période d'instabilité.** Les trois stablecoins s'écartent fortement de 1 \$.
  Le DAI atteint un pic de près de **1,09 \$** en mars 2020 (krach du Covid), et l'USDC
  oscille entre **0,97 \$ et 1,04 \$**. Le marché des stablecoins était alors encore jeune
  et peu liquide. Le DAI reste agité tout au long de 2020, avec une prime au-dessus de 1 \$
  liée au boom de la finance décentralisée (DeFi).
- **2021-2022 : stabilisation progressive.** Les écarts se réduisent nettement, avec
  quelques secousses ponctuelles, notamment pour le DAI fin 2021 et l'USDT mi-2022,
  après l'effondrement de Terra.
- **Mars 2023 : la crise SVB.** L'USDC chute brutalement, et le **DAI chute avec lui**,
  car une large part de ses garanties était constituée d'USDC : c'est un effet de
  **contagion**. À l'inverse, l'USDT monte légèrement au-dessus de 1 \$, les investisseurs
  s'y réfugiant.
- **2024-2026 : une grande stabilité.** Les trois courbes sont quasiment collées au peg.

In [10]:
fig = px.line(ecart,
              title="Écart au peg (en points de base)",
              labels={"value": "Écart (pb)", "Date": "Date", "variable": "Stablecoin"})

fig.add_hline(y=50, line_dash="dot", line_color="red")    # seuil d'alerte haut
fig.add_hline(y=-50, line_dash="dot", line_color="red")   # seuil d'alerte bas
fig.show()

**Interprétation du graphique 2 :**
L'expression en points de base rend les chocs plus lisibles :
- les plus forts écarts datent de 2020 : environ **+930 pb pour le DAI** (soit +9,3 %),
  ainsi que **+440 pb et −300 pb pour l'USDC** ;
- lors de la crise SVB (mars 2023), l'USDC et le DAI décrochent d'environ
  **−290 pb**, alors que l'USDT dépasse le peg d'environ **+70 pb** ;
- depuis 2021, en dehors de ces épisodes, les écarts restent presque toujours
  **à l'intérieur du seuil d'alerte de ±50 pb**.

Les stablecoins ne sont donc pas « toujours stables » : ils sont stables
**la plupart du temps**, avec des crises **rares, courtes et violentes**.

In [11]:
ecart_abs = ecart.abs()   # valeur absolue : on ignore le sens de l'écart

tableau_ecarts = pd.DataFrame({
    "% jours > 10 pb":  (ecart_abs > 10).mean() * 100,
    "% jours > 50 pb":  (ecart_abs > 50).mean() * 100,
    "% jours > 100 pb": (ecart_abs > 100).mean() * 100,
}).round(2)

tableau_ecarts

,% jours > 10 pb,% jours > 50 pb,% jours > 100 pb
USDT,19.25,1.28,0.36
USDC,9.38,2.25,1.24
DAI,22.77,9.66,5.41


**Interprétation du tableau de fréquence :**

| | > 10 pb | > 50 pb | > 100 pb |
|---|---|---|---|
| **USDT** | 19,25 % | 1,28 % (≈ 32 jours) | 0,36 % (≈ 9 jours) |
| **USDC** | 9,39 % | 2,25 % (≈ 56 jours) | 1,24 % (≈ 31 jours) |
| **DAI**  | 22,78 % | 9,67 % (≈ 241 jours) | 5,42 % (≈ 135 jours) |

- **L'USDC est le plus précis au quotidien :** il ne s'écarte de plus de 10 pb que
  sur 9 % des jours, contre environ 20 % pour l'USDT et le DAI.
- **Mais l'USDT résiste mieux aux grands chocs :** il ne dépasse 100 pb que sur
  0,36 % des jours, contre 1,24 % pour l'USDC. L'USDT connaît de petits écarts
  fréquents mais rarement de grands décrochages.
- **Le DAI est clairement le moins stable :** il dépasse 100 pb sur plus de 5 % des jours,
  ce qui confirme la fragilité d'un stablecoin garanti par des cryptoactifs volatils.

**Conclusion intermédiaire :** la stabilité d'un stablecoin a deux dimensions,
la **précision quotidienne** et la **résistance aux crises extrêmes**.
Aucun des trois n'excelle sur les deux à la fois. Pour un stablecoin dirham,
un modèle adossé à des réserves en monnaie (type USDC/USDT) est nettement
préférable à un modèle garanti par des cryptoactifs (type DAI).

In [12]:
avant = ecart_abs.loc[:"2020-12-31"]     # uniquement l'année 2020
apres = ecart_abs.loc["2021-01-01":]     # de 2021 à aujourd'hui

comparaison = pd.DataFrame({
    "2020 : % jours > 50 pb":   (avant > 50).mean() * 100,
    "2021+ : % jours > 50 pb":  (apres > 50).mean() * 100,
    "2020 : écart moyen (pb)":  avant.mean(),
    "2021+ : écart moyen (pb)": apres.mean(),
}).round(2)

comparaison

,2020 : % jours > 50 pb,2021+ : % jours > 50 pb,2020 : écart moyen (pb),2021+ : écart moyen (pb)
USDT,7.92,0.14,22.51,4.71
USDC,14.48,0.14,34.29,1.95
DAI,62.57,0.56,98.71,4.61


**Interprétation : la maturation du marché des stablecoins**

| | 2020 : jours > 50 pb | 2021+ : jours > 50 pb | 2020 : écart moyen | 2021+ : écart moyen |
|---|---|---|---|---|
| **USDT** | 7,92 % (≈ 29 jours) | 0,14 % (≈ 3 jours) | 22,5 pb | 4,7 pb |
| **USDC** | 14,48 % (≈ 53 jours) | 0,14 % (≈ 3 jours) | 34,3 pb | 1,9 pb |
| **DAI**  | 62,57 % (≈ 229 jours) | 0,56 % (≈ 12 jours) | 98,7 pb | 4,6 pb |

- **Une rupture nette entre 2020 et la suite.** Les écarts importants (> 50 pb) deviennent
  quasi inexistants à partir de 2021 : environ 3 jours en près de 6 ans pour l'USDT et
  l'USDC. L'écart moyen est divisé par **5 pour l'USDT**, **18 pour l'USDC** et **21 pour le DAI**.
- **Cette amélioration s'explique par la maturité du marché :** forte hausse des volumes
  échangés, multiplication des plateformes d'échange, arbitrage plus efficace et
  meilleure transparence sur les réserves.
- **La hiérarchie change sur la période récente :** depuis 2021, l'**USDC est le plus
  précis** (1,9 pb d'écart moyen, soit 2,5 fois moins que l'USDT). Le constat
  précédent, selon lequel l'USDT résistait mieux aux grands chocs, était surtout dû
  à l'année 2020.
- **Le DAI reste le moins stable,** même s'il a fortement progressé.

**Choix méthodologique pour la suite :** l'année 2020 reflète un marché immature et
ne représente pas les conditions actuelles. Dans la suite de l'analyse, la période
**2021-2026** servira de référence pour mesurer le risque « normal », tandis que les
crises (2020, Terra 2022, SVB 2023) seront étudiées séparément comme **scénarios de stress**.

**Conclusion de la partie 3 :** les stablecoins adossés à des réserves en monnaie
(USDC, USDT) maintiennent aujourd'hui leur peg avec une grande précision, à quelques
points de base près. Leur risque principal n'est plus l'instabilité quotidienne mais
les **chocs de confiance soudains**, comme la crise SVB, que nous analyserons
dans la partie suivante.

## 4. Mesure du risque de marché

### 4.1 Rendements journaliers et volatilité

Pour mesurer le risque, nous analysons les **rendements journaliers**, c'est-à-dire
la variation du prix d'un jour à l'autre :

> Rendement = (Prix du jour ÷ Prix de la veille) − 1

Conformément au choix méthodologique de la partie 3, l'analyse porte sur la
période **2021-2026**, représentative du marché actuel.

La **volatilité** (écart-type des rendements) mesure l'amplitude des variations.
Elle est annualisée en multipliant par √365, les cryptoactifs étant échangés
7 jours sur 7. Le **Bitcoin** sert de point de comparaison.

In [13]:
# Garder seulement la période 2021-2026
prix_recent = prix.loc["2021-01-01":]

# Rendement journalier : variation en % par rapport à la veille
rendements = prix_recent.pct_change().dropna()

print("✅ Rendements calculés :", rendements.shape)

✅ Rendements calculés : (2127, 3)


**Explication du code :**
Nous calculons le rendement journalier de chaque stablecoin sur la période
2021-2026. La première observation est supprimée car elle n'a pas de jour précédent.
Le tableau `rendements` servira de base à tous les calculs de risque de cette partie.

In [14]:
# Télécharger le Bitcoin sur la même période (référence)
btc = yf.Ticker("BTC-USD").history(start="2021-01-01")["Close"]
btc.index = btc.index.tz_localize(None)
rend_btc = btc.pct_change().dropna()

# Tableau de volatilité
volatilite = pd.DataFrame({
    "Volatilité journalière (%)": rendements.std() * 100,
    "Volatilité annualisée (%)":  rendements.std() * np.sqrt(365) * 100,
})

# Ajouter une ligne pour le Bitcoin
volatilite.loc["BTC (référence)"] = [rend_btc.std() * 100,
                                     rend_btc.std() * np.sqrt(365) * 100]

volatilite.round(3)

,Volatilité journalière (%),Volatilité annualisée (%)
USDT,0.053,1.018
USDC,0.089,1.698
DAI,0.124,2.370
BTC (référence),2.968,56.696


**Interprétation :**

| | Volatilité journalière | Volatilité annualisée | Bitcoin est … fois plus volatil |
|---|---|---|---|
| **USDT** | 0,053 % | 1,02 % | ≈ 56 × |
| **USDC** | 0,089 % | 1,70 % | ≈ 33 × |
| **DAI**  | 0,124 % | 2,37 % | ≈ 24 × |
| **BTC**  | 2,968 % | 56,71 % | référence |

- **Les stablecoins sont 24 à 56 fois moins volatils que le Bitcoin**, ce qui confirme
  qu'ils remplissent leur rôle de réserve de valeur stable au sein de l'écosystème crypto.
- **L'USDT est le moins volatil** (1,02 % par an), suivi de l'USDC (1,70 %) et du DAI (2,37 %).
  Le DAI reste le plus risqué, ce qui est cohérent avec sa garantie en cryptoactifs.
- **Précision et volatilité ne mesurent pas la même chose.** L'USDC est le plus proche
  de 1 \$ en moyenne (partie 3), mais il est plus volatil que l'USDT. Cela s'explique par
  la nature de ses écarts : l'USDT présente de légers écarts permanents mais peu de
  variations d'un jour à l'autre, tandis que l'USDC, habituellement collé au peg, a
  connu un choc brutal et isolé (crise SVB de mars 2023) qui pèse fortement dans l'écart-type.

**Limite :** l'annualisation par √365 suppose que les variations s'accumulent au fil
du temps. Or le prix d'un stablecoin est **ramené vers 1 \$** par les mécanismes
d'arbitrage. La volatilité annualisée surestime donc le risque réel sur un an et
sert surtout ici d'**outil de comparaison** avec d'autres actifs.

### 4.2 Value at Risk (VaR) et Expected Shortfall (ES)

Nous nous plaçons dans le cas d'une entreprise qui détient **1 000 000 \$** de trésorerie
en stablecoins, et nous mesurons sa perte potentielle sur une journée.

- **VaR 95 % / 99 %** : perte journalière qui n'est dépassée que dans 5 % / 1 % des cas.
- **Expected Shortfall 99 %** : perte moyenne observée lors des 1 % pires journées.
  Plus exigeante que la VaR, c'est la mesure retenue par le régulateur bancaire
  (accords de Bâle) pour capter le risque de queue.

La méthode utilisée est la **VaR historique** : les pertes sont tirées directement
des rendements observés sur 2021-2026, sans hypothèse sur leur distribution.
Le Bitcoin est ajouté comme référence.

In [15]:
montant = 1_000_000   # trésorerie de 1 million de dollars

# Ajouter le Bitcoin aux rendements pour comparer
tous = rendements.copy()
tous["BTC"] = rend_btc

resultats = {}

for nom in tous.columns:
    r = tous[nom].dropna()

    q95 = r.quantile(0.05)          # limite des 5 % pires jours
    q99 = r.quantile(0.01)          # limite des 1 % pires jours
    es99 = r[r <= q99].mean()       # moyenne des 1 % pires jours

    resultats[nom] = {
        "VaR 95 % ($)": -q95 * montant,
        "VaR 99 % ($)": -q99 * montant,
        "ES 99 % ($)":  -es99 * montant,
    }

risque = pd.DataFrame(resultats).T.round(0)
risque

,VaR 95 % ($),VaR 99 % ($),ES 99 % ($)
USDT,605.0,1175.0,2225.0
USDC,413.0,961.0,3124.0
DAI,870.0,2665.0,6257.0
BTC,45355.0,78517.0,107841.0


**Interprétation (pour une trésorerie de 1 000 000 \$, horizon 1 jour) :**

- **Les stablecoins présentent un risque journalier très faible.** La VaR 99 % de l'USDC
  est de 961 \$, soit moins de 0,1 % du capital. À titre de comparaison, la même
  somme placée en Bitcoin pourrait perdre 78 523 \$ (7,9 %) lors d'un jour sur cent,
  soit un risque environ **80 fois supérieur**.

- **Selon la VaR, l'USDC est le stablecoin le moins risqué** (VaR 95 % de 413 \$ et
  VaR 99 % de 961 \$), devant l'USDT et le DAI.

- **Mais l'Expected Shortfall inverse le classement entre USDC et USDT.** Lors des 1 %
  pires journées, l'USDC perd en moyenne **3 124 \$**, contre **2 225 \$** pour l'USDT.
  Le rapport ES / VaR met en évidence l'épaisseur de la queue de distribution :

  | | ES 99 % / VaR 99 % |
  |---|---|
  | USDT | 1,9 |
  | USDC | **3,3** |
  | DAI | 2,3 |
  | BTC | 1,4 |

  Chez l'USDC, les pertes extrêmes sont plus de trois fois supérieures à la VaR :
  ses rares chocs (crise SVB) sont beaucoup plus violents que ses mauvais jours ordinaires.

- **Le DAI reste le plus risqué des trois stablecoins** sur tous les indicateurs.

**Conclusion intermédiaire :** la VaR seule aurait désigné l'USDC comme le plus sûr.
L'Expected Shortfall révèle au contraire un **risque de queue** significatif, lié
aux chocs de confiance. Ce résultat illustre pourquoi le régulateur bancaire privilégie
l'ES, et montre que le principal danger d'un stablecoin n'est pas la fluctuation
quotidienne, mais la **crise de confiance soudaine**.

In [16]:
# Rendements de l'USDC en points de base
r_usdc = rendements["USDC"] * 10000

var99 = -r_usdc.quantile(0.01)
es99  = -r_usdc[r_usdc <= r_usdc.quantile(0.01)].mean()

fig = px.histogram(r_usdc, nbins=200, log_y=True,
                   title="Distribution des rendements journaliers de l'USDC (2021-2026)",
                   labels={"value": "Rendement journalier (pb)"})

# Lignes verticales pour la VaR et l'ES
fig.add_vline(x=-var99, line_dash="dash", line_color="orange",
              annotation_text="VaR 99 %", annotation_position="top left")
fig.add_vline(x=-es99, line_dash="dash", line_color="red",
              annotation_text="ES 99 %", annotation_position="bottom left")

fig.update_layout(showlegend=False, yaxis_title="Nombre de jours (échelle log)")
fig.show()

**Interprétation de l'histogramme :**

- **Une concentration extrême autour de zéro.** La grande majorité des journées présentent
  un rendement compris entre −5 et +5 pb : en temps normal, l'USDC ne varie quasiment pas.
- **La VaR 99 % (≈ −10 pb) et l'ES 99 % (≈ −31 pb) restent proches du centre**, car elles
  sont calculées sur l'ensemble des jours, dominés par les périodes calmes.
- **Quelques journées isolées sont très éloignées du centre :** environ −280 pb et −100 pb
  à gauche, +65, +95 et +210 pb à droite. Chacune correspond à **un seul jour** en près
  de six ans. Le pire jour (≈ −280 pb) et le meilleur (≈ +210 pb) correspondent très
  probablement à la crise SVB de mars 2023 : une chute brutale suivie d'un rebond
  tout aussi violent après l'intervention des autorités américaines.

**Lecture en termes de risque :** la distribution présente des **queues épaisses**
(*fat tails*) : les événements extrêmes sont rares mais d'une ampleur sans commune
mesure avec les variations habituelles. Le pire jour représente une perte environ
**28 fois supérieure à la VaR 99 %**. Un modèle fondé sur la loi normale sous-estimerait
fortement ce risque, d'où l'intérêt de la méthode historique et de l'Expected Shortfall.

## 5. Étude des crises historiques

Nous analysons le comportement des trois stablecoins lors de trois épisodes de stress :
- **Covid-19 (mars 2020)** : krach généralisé des marchés financiers ;
- **Terra (mai 2022)** : effondrement du stablecoin algorithmique UST, dépourvu de
  réserves réelles, entraînant une perte d'environ 40 milliards de dollars et
  une contagion à l'ensemble du marché ;
- **SVB (mars 2023)** : faillite de Silicon Valley Bank, où étaient déposés
  3,3 milliards de dollars des réserves de l'USDC.

Pour chaque crise, trois indicateurs sont mesurés :
- **Profondeur** : prix le plus bas atteint (en clôture et en intrajournalier) ;
- **Durée** : nombre de jours où l'écart au peg dépasse 50 pb ;
- **Récupération** : nombre de jours nécessaires pour revenir à moins de 10 pb du peg.

In [17]:
series_low = []

for nom, code in tickers.items():
    historique = yf.Ticker(code).history(start="2020-01-01")
    series_low.append(historique["Low"].rename(nom))   # garder le prix le plus bas du jour

bas = pd.concat(series_low, axis=1)
bas.index = bas.index.tz_localize(None)

print("✅ Prix les plus bas téléchargés :", bas.shape)

✅ Prix les plus bas téléchargés : (2494, 3)


In [18]:
# Les 3 crises : nom -> (date de début, date de fin)
crises = {
    "Covid (mars 2020)": ("2020-03-01", "2020-04-30"),
    "Terra (mai 2022)":  ("2022-05-01", "2022-06-15"),
    "SVB (mars 2023)":   ("2023-03-01", "2023-03-31"),
}

# La "recette" qui analyse une crise
def analyser_crise(nom_crise, debut, fin):
    lignes = []
    for coin in prix.columns:
        cloture  = prix[coin].loc[debut:fin]     # prix de clôture pendant la crise
        plus_bas = bas[coin].loc[debut:fin]      # prix les plus bas pendant la crise

        # 1. Profondeur : le pire jour
        date_pire = cloture.idxmin()

        # 2. Durée : nombre de jours avec un écart > 50 pb (0,005 $)
        jours_hors_peg = ((cloture - 1).abs() > 0.005).sum()

        # 3. Récupération : premier jour après le pire où l'écart revient sous 10 pb
        apres  = cloture.loc[date_pire:]
        retour = apres[(apres - 1).abs() <= 0.001]
        jours_recup = (retour.index[0] - date_pire).days if len(retour) > 0 else None

        lignes.append({
            "Crise": nom_crise,
            "Stablecoin": coin,
            "Plus bas intrajournalier ($)": round(plus_bas.min(), 4),
            "Pire clôture ($)": round(cloture.min(), 4),
            "Date pire clôture": date_pire.date(),
            "Jours hors ±50 pb": jours_hors_peg,
            "Jours pour revenir à ±10 pb": jours_recup,
        })
    return lignes

# Appliquer la recette aux 3 crises
toutes = []
for nom_crise, (debut, fin) in crises.items():
    toutes += analyser_crise(nom_crise, debut, fin)

tableau_crises = pd.DataFrame(toutes).set_index(["Crise", "Stablecoin"])
tableau_crises

Plus bas intrajournalier ($)  Pire clôture ($)  \
Crise             Stablecoin                                                   
Covid (mars 2020) USDT                              0.8995            0.9742   
                  USDC                              0.9292            0.9701   
                  DAI                               0.9640            0.9713   
Terra (mai 2022)  USDT                              0.9485            0.9959   
                  USDC                              0.9993            0.9996   
                  DAI                               0.9942            0.9983   
SVB (mars 2023)   USDT                              0.9991            1.0000   
                  USDC                              0.8774            0.9715   
                  DAI                               0.8970            0.9739   

                             Date pire clôture  Jours hors ±50 pb  \
Crise             Stablecoin                                        
Covid (mars 2020) USDT              2020-03-18                 15   
                  USDC              2020-03-18                 15   
                  DAI               2020-03-22                 43   
Terra (mai 2022)  USDT              2022-05-11                  0   
                  USDC              2022-05-09                  0   
                  DAI               2022-05-09                  0   
SVB (mars 2023)   USDT              2023-03-09                  2   
                  USDC              2023-03-11                  2   
                  DAI               2023-03-11                  2   

                              Jours pour revenir à ±10 pb  
Crise             Stablecoin                               
Covid (mars 2020) USDT                                  3  
                  USDC                                  3  
                  DAI                                   5  
Terra (mai 2022)  USDT                                  4  
                  USDC                                  0  
                  DAI                                   1  
SVB (mars 2023)   USDT                                  0  
                  USDC                                  3  
                  DAI                                   4

**Interprétation du tableau des crises :**

| Crise | Stablecoin le plus touché | Plus bas intrajournalier | Récupération (±10 pb) |
|---|---|---|---|
| Covid (mars 2020) | USDT | 0,8995 \$ | 3 jours |
| Terra (mai 2022) | USDT | 0,9485 \$ | 4 jours |
| SVB (mars 2023) | USDC | 0,8774 \$ | 3 jours |

**1. Chaque crise frappe un stablecoin différent, selon la nature du choc.**
- **Covid (2020)** : choc **systémique**. Les trois stablecoins décrochent simultanément
  dans un marché encore peu liquide. Le DAI reste 43 jours hors du seuil de ±50 pb,
  principalement en raison d'une prime au-dessus de 1 \$.
- **Terra (2022)** : choc de **confiance** sur les réserves. L'USDT, critiqué pour
  son manque de transparence, tombe à 0,9485 \$ en séance (−5,2 %), tandis que l'USDC
  reste quasiment immobile (0,9993 \$).
- **SVB (2023)** : choc de **contrepartie**. Les réserves de l'USDC étant en partie
  bloquées dans une banque en faillite, il chute à 0,8774 \$ en séance (−12,3 %),
  entraînant le DAI par **contagion** (0,8970 \$). L'USDT, à l'inverse, joue le rôle
  de **valeur refuge** et passe brièvement au-dessus de son peg.

**2. Les crises sont violentes mais brèves.** Dans tous les cas, le peg est rétabli en
**5 jours au plus**, grâce aux mécanismes de rachat 1 pour 1 et, dans le cas de SVB,
à la garantie des dépôts annoncée par les autorités américaines. Ce constat contraste
avec l'UST de Terra, stablecoin algorithmique sans réserves réelles, qui ne s'est
jamais redressé.

**3. Les données de clôture sous-estiment fortement les chocs.** Lors de la crise SVB,
la pire clôture de l'USDC (0,9715 \$) masque un plus bas intrajournalier de 0,8774 \$,
soit un écart d'environ 940 pb. Une mesure du risque fondée uniquement sur les
clôtures sous-évalue donc la gravité réelle des crises.

**Limite :** l'indicateur de récupération est calculé à partir du prix le plus bas.
Il n'est donc pas pertinent pour les écarts **à la hausse**, comme celui de l'USDT
pendant la crise SVB.

**Enseignements pour un stablecoin dirham :** la résilience repose sur trois piliers :
des **réserves réelles et liquides**, une **transparence** totale sur leur composition
et une **diversification des banques dépositaires**, afin d'éviter un scénario de type SVB.

In [19]:
for nom_crise, (debut, fin) in crises.items():
    fig = px.line(prix.loc[debut:fin],
                  title=f"Zoom sur la crise : {nom_crise}",
                  labels={"value": "Prix de clôture ($)", "Date": "Date", "variable": "Stablecoin"})

    fig.add_hline(y=1, line_dash="dash", line_color="black")        # le peg
    fig.add_hline(y=1.005, line_dash="dot", line_color="red")       # seuil +50 pb
    fig.add_hline(y=0.995, line_dash="dot", line_color="red")       # seuil -50 pb
    fig.show()


**Interprétation des graphiques de crise :**

**Covid (mars 2020) : un choc systémique dans les deux sens.**
Lors du krach du 12-13 mars, les investisseurs fuient les cryptoactifs vers les
stablecoins, qui passent au-dessus de leur peg (DAI ≈ 1,09 \$, USDT ≈ 1,05 \$,
USDC ≈ 1,04 \$). Le 17-18 mars, le mouvement s'inverse et l'USDT et l'USDC tombent
autour de 0,97 \$. Le DAI reste instable jusqu'à fin avril et dépasse régulièrement
le seuil de ±50 pb, le plus souvent au-dessus de 1 \$.

**Terra (mai 2022) : une crise de confiance ciblée sur l'USDT.**
*(L'échelle de ce graphique est très resserrée : 0,996 \$ à 1,004 \$.)*
L'USDT chute le 12 mai à environ 0,9959 \$ en clôture (plus bas intrajournalier :
0,9485 \$), puis reste légèrement sous 1 \$ pendant plus d'un mois, signe d'une méfiance
persistante envers ses réserves. À l'inverse, l'USDC et le DAI se maintiennent
légèrement au-dessus du peg : les investisseurs se tournent vers les stablecoins
jugés plus transparents (**fuite vers la qualité**).

**SVB (mars 2023) : un choc de contrepartie en forme de « V ».**
Le 11 mars, l'USDC (≈ 0,9715 \$) et le DAI (≈ 0,974 \$) chutent ensemble : leurs courbes
quasi superposées illustrent la **contagion**, le DAI étant largement garanti par
de l'USDC. Dès l'annonce de la garantie des dépôts par les autorités américaines
le 12 mars, ils reviennent près de 1 \$ en deux jours. L'USDT évolue en miroir
(jusqu'à ≈ 1,0075 \$) et joue le rôle de **valeur refuge**.

## 6. Cas d'usage : les transferts des Marocains résidant à l'étranger (MRE)

Les transferts des MRE ont dépassé **122 milliards de dirhams en 2025** (Office des changes),
ce qui en fait l'une des premières sources de devises du Maroc. Leur coût représente
donc un enjeu majeur pour les familles bénéficiaires.

Le coût d'un transfert comprend deux éléments :
- les **frais affichés** par le prestataire ;
- la **perte sur le taux de change**, coût implicite non affiché.

**Données utilisées :** relevés de la Banque mondiale (*Remittance Prices Worldwide*)
au 3e trimestre 2025, pour un envoi de **140 €** (équivalent de 200 \$) au départ de la France.

**Hypothèse stablecoin :** le circuit par stablecoin n'étant pas encore encadré au Maroc,
nous modélisons un futur stablecoin émis par un établissement agréé, avec un coût total
de **1 %** (achat ≈ 0,5 % + transfert quasi gratuit + conversion en dirhams ≈ 0,5 %).
Cette hypothèse fera l'objet d'une analyse de sensibilité.

**Référence :** l'Objectif de développement durable n°10 des Nations unies vise un coût
inférieur à **3 %** d'ici 2030.

*Sources : Office des changes ; Banque mondiale, Remittance Prices Worldwide (T3 2025),
relayé par JeChange (octobre 2026).*

In [20]:
# Coûts moyens au départ de la France (Banque mondiale, T3 2025) + hypothèse stablecoin
canaux = pd.DataFrame({
    "Canal": ["Banque (virement international)",
              "Banque via opérateur partenaire",
              "Opérateur de transfert",
              "Paiement mobile",
              "Stablecoin (hypothèse)"],
    "Coût total (%)": [18.59, 4.58, 4.09, 2.27, 1.00],
    "Délai indicatif": ["1 à 6 jours", "Le jour même", "Moins d'une heure",
                        "Quelques minutes", "Quelques minutes, 24h/24"],
})

montant_envoi = 140   # euros, montant type de la Banque mondiale
envois_par_an = 12    # une famille qui envoie de l'argent chaque mois

canaux["Frais par envoi (€)"] = (canaux["Coût total (%)"] / 100 * montant_envoi).round(2)
canaux["Frais par an (€)"]    = (canaux["Frais par envoi (€)"] * envois_par_an).round(2)

canaux

,Canal,Coût total (%),Délai indicatif,Frais par envoi (€),Frais par an (€)
0,Banque (virement international),18.59,1 à 6 jours,26.03,312.36
1,Banque via opérateur partenaire,4.58,Le jour même,6.41,76.92
2,Opérateur de transfert,4.09,Moins d'une heure,5.73,68.76
3,Paiement mobile,2.27,Quelques minutes,3.18,38.16
4,Stablecoin (hypothèse),1.00,"Quelques minutes, 24h/24",1.40,16.80


**Interprétation du tableau :**

Pour une famille qui envoie **140 € par mois** (12 envois par an) :

| Canal | Frais par an | Écart avec le stablecoin |
|---|---|---|
| Banque (virement international) | 312,36 € | + 295,56 € |
| Banque via opérateur partenaire | 76,92 € | + 60,12 € |
| Opérateur de transfert | 68,76 € | + 51,96 € |
| Paiement mobile | 38,16 € | + 21,36 € |
| **Stablecoin (hypothèse)** | **16,80 €** | — |

- **Le virement bancaire international est de loin le canal le plus coûteux** : 18,59 %,
  soit plus de 300 € de frais par an, l'équivalent de plus de deux envois perdus.
- **Le bon point de comparaison est l'opérateur de transfert**, qui concentre la grande
  majorité des offres (221 sur 256 relevées au départ de la France). Face à lui, un stablecoin
  à 1 % permettrait d'économiser environ **52 € par an et par famille (−76 %)**.
- **Le stablecoin cumule coût bas et rapidité** : envoi en quelques minutes, 24h/24, y compris
  le week-end, alors que le virement bancaire peut prendre jusqu'à 6 jours.

*Ces résultats reposent sur l'hypothèse d'un coût de 1 % pour le stablecoin,
testée dans l'analyse de sensibilité qui suit.*

In [21]:
fig = px.bar(canaux, x="Canal", y="Coût total (%)", text="Coût total (%)",
             title="Coût d'un envoi de 140 € de la France vers le Maroc, selon le canal",
             color="Canal")

fig.update_traces(texttemplate="%{text:.2f} %", textposition="outside")   # valeur au-dessus des barres

fig.add_hline(y=5.23, line_dash="dot", line_color="gray",
              annotation_text="Moyenne du marché : 5,23 %", annotation_position="top left")
fig.add_hline(y=3, line_dash="dash", line_color="green",
              annotation_text="Objectif ONU 2030 : 3 %", annotation_position="bottom left")

fig.update_layout(showlegend=False, yaxis_range=[0, 21])   # un peu d'espace en haut
fig.show()

**Interprétation du graphique :**
- Seuls **deux canaux respectent l'objectif de l'ONU de 3 %** : le paiement mobile (2,27 %)
  et le stablecoin (1 %, hypothèse).
- Les opérateurs de transfert (4,09 %) et les banques via partenaire (4,58 %) se situent
  sous la moyenne du marché (5,23 %), mais restent au-dessus de l'objectif.
- Le virement bancaire (18,59 %) est **plus de trois fois plus cher que la moyenne**
  et environ **18 fois plus cher** que le stablecoin.

Le stablecoin apparaît ainsi comme un levier potentiel pour atteindre l'objectif
international de réduction du coût des transferts, sous réserve d'un cadre réglementaire
permettant une conversion simple et peu coûteuse en dirhams.

### 6.2 Impact potentiel à l'échelle nationale

Nous estimons les économies que permettrait un stablecoin dirham sur l'ensemble
des transferts des MRE (122 milliards de DH en 2025), selon deux paramètres :
- la **part des transferts** réalisés en stablecoin (5 %, 10 %, 25 %, 50 %) ;
- le **coût du stablecoin** (0,5 %, 1 % et 2 %), pour tester la robustesse du résultat.

> Économie = Transferts × Part en stablecoin × (Coût actuel − Coût stablecoin)

Le coût actuel retenu est le coût moyen relevé au départ de la France (5,23 %).

In [22]:
transferts_mre = 122e9        # 122 milliards de DH (2025, Office des changes)
cout_actuel    = 5.23 / 100   # coût moyen actuel (Banque mondiale, France)

frais_actuels = transferts_mre * cout_actuel
print(f"💸 Frais payés estimés aujourd'hui : {frais_actuels / 1e9:.2f} milliards de DH par an\n")

adoptions        = [5, 10, 25, 50]       # part des transferts en stablecoin (%)
couts_stablecoin = [0.5, 1.0, 2.0]       # coût du stablecoin (%)

lignes = []
for cout_sc in couts_stablecoin:            # pour chaque coût possible
    for adoption in adoptions:              # pour chaque niveau d'adoption
        economie = transferts_mre * (adoption / 100) * (cout_actuel - cout_sc / 100)
        lignes.append({
            "Adoption (%)": adoption,
            "Coût stablecoin": f"{cout_sc} %",
            "Économie (milliards DH)": round(economie / 1e9, 2),
        })

sensibilite = pd.DataFrame(lignes)

# Mettre en forme : adoption en lignes, coût en colonnes
sensibilite.pivot(index="Adoption (%)", columns="Coût stablecoin",
                  values="Économie (milliards DH)")

💸 Frais payés estimés aujourd'hui : 6.38 milliards de DH par an



Coût stablecoin,0.5 %,1.0 %,2.0 %
Adoption (%),,,
5,0.29,0.26,0.20
10,0.58,0.52,0.39
25,1.44,1.29,0.99
50,2.89,2.58,1.97


**Interprétation :**

**1. Le coût actuel des transferts est considérable.** Avec un coût moyen de 5,23 %,
les frais supportés sur les 122 milliards de DH de transferts des MRE sont estimés à
environ **6,38 milliards de DH par an**, autant de ressources qui n'atteignent pas
les familles bénéficiaires.

**2. Les économies potentielles sont significatives dans tous les scénarios.**

| Part des transferts en stablecoin | Économie annuelle (coût de 0,5 % à 2 %) |
|---|---|
| 5 % | 0,20 à 0,29 milliard de DH |
| 10 % | 0,39 à 0,58 milliard de DH |
| 25 % | 0,99 à 1,44 milliard de DH |
| 50 % | 1,97 à 2,89 milliards de DH |

Dans le scénario central (adoption de 50 % et coût de 1 %), l'économie atteint
**2,58 milliards de DH par an**, soit une baisse d'environ **40 %** des frais
actuellement payés.

In [23]:
fig = px.bar(sensibilite, x="Adoption (%)", y="Économie (milliards DH)",
             color="Coût stablecoin", barmode="group", text="Économie (milliards DH)",
             title="Économies annuelles potentielles pour les familles marocaines (milliards de DH)")

fig.update_traces(textposition="outside")
fig.update_xaxes(type="category")    # 5, 10, 25, 50 affichés comme des catégories
fig.show()

**Interprétation du graphique :**
- **L'économie croît proportionnellement au taux d'adoption** : elle est multipliée par 10
  entre 5 % et 50 % d'adoption.
- **Le coût du stablecoin a un effet beaucoup plus limité** : à 50 % d'adoption,
  l'économie varie seulement entre 1,97 et 2,89 milliards de DH selon que le coût
  est de 2 % ou de 0,5 %.
- **Le résultat est robuste** : même dans l'hypothèse la plus prudente (coût de 2 %),
  le stablecoin reste nettement moins cher que les canaux actuels.

**Enseignement clé :** le principal déterminant des gains n'est pas la technologie
elle-même, mais le **niveau d'adoption**, qui dépend de la confiance des utilisateurs,
de la simplicité d'usage et du cadre réglementaire.

### Conclusion de la partie 6

Les transferts des MRE constituent un **cas d'usage concret et à fort impact** pour
un stablecoin dirham. Le coût actuel (5,23 % en moyenne, jusqu'à 18,59 % par virement
bancaire) représente environ 6,4 milliards de DH par an. Un stablecoin régulé pourrait
réduire ce coût de plusieurs milliards de dirhams, tout en rendant les transferts
instantanés et disponibles 24h/24.

**Limites de l'analyse :**
- le coût moyen retenu (5,23 %) concerne le corridor **France–Maroc**. Les corridors
  Espagne, Italie ou pays du Golfe peuvent présenter des coûts différents ;
- une partie des transferts transite par des **canaux informels** (espèces transportées
  lors des séjours au Maroc), non concernés par ces frais ;
- le coût du stablecoin est une **hypothèse**, le circuit n'étant pas encore encadré
  au Maroc ;
- les économies des familles correspondent à un **manque à gagner** pour les banques
  et opérateurs de transfert actuels.

Ce dernier point est stratégique : une **banque marocaine** qui émettrait elle-même un
stablecoin dirham pourrait compenser cette perte de commissions par les **revenus tirés
de ses réserves**. C'est précisément ce modèle économique que nous étudions dans
la partie suivante.

## 7. Business plan du « MADc », stablecoin adossé au dirham

Nous modélisons un stablecoin fictif, le **MADc** (1 MADc = 1 DH), émis par une banque
marocaine agréée dans le cadre du futur cadre légal. Chaque MADc émis est couvert à 100 %
par une réserve en dirhams.

**Sources de revenus :**
1. **Intérêts sur les réserves**, placées en bons du Trésor marocain ;
2. **Commissions** de 0,5 % sur les transferts des MRE réalisés en MADc.

**Principales hypothèses :**

| Hypothèse | Valeur |
|---|---|
| Encours (années 1 à 5) | 1 → 3 → 6 → 9 → 12 milliards de DH |
| Allocation des réserves | 20 % liquidités, 50 % BT 13 semaines, 30 % BT 52 semaines |
| Taux BT 13 / 52 semaines | 2,170 % / 2,363 % (adjudication du 20/10/2026) |
| Part des transferts MRE en MADc | 2 % → 5 % → 10 % → 15 % → 20 % (sur 122 Mds DH) |
| Commission de l'émetteur | 0,5 % |
| Coûts fixes annuels | technologie 30, conformité 20, personnel 25 M DH ; marketing 15 puis 10 M DH |
| Coûts variables | 0,1 % du volume de transferts |
| Investissement initial | 80 M DH |
| Impôt sur les sociétés | 40 % (établissements de crédit, 2026) |

*Contexte : taux directeur de Bank Al-Maghrib maintenu à 2,25 % (septembre 2026).*

In [24]:
# ---------- HYPOTHÈSES DU MODÈLE ----------
annees = [1, 2, 3, 4, 5]

# Encours de MADc en circulation (milliards de DH, fin d'année)
encours = [1, 3, 6, 9, 12]

# Placement des réserves : (part, taux)
part_cash, taux_cash = 0.20, 0.0        # liquidités : disponibles mais ne rapportent rien
part_bt13, taux_bt13 = 0.50, 0.02170    # bons du Trésor 13 semaines
part_bt52, taux_bt52 = 0.30, 0.02363    # bons du Trésor 52 semaines

rendement_reserves = part_cash * taux_cash + part_bt13 * taux_bt13 + part_bt52 * taux_bt52

# Transferts des MRE passant par le MADc
transferts_mre = 122                          # milliards de DH par an
part_mre = [0.02, 0.05, 0.10, 0.15, 0.20]     # adoption progressive
commission = 0.005                            # 0,5 % prélevé par l'émetteur

# Coûts (millions de DH par an)
cout_tech       = 30
cout_conformite = 20
cout_personnel  = 25
cout_marketing  = [15, 15, 10, 10, 10]
cout_variable   = 0.001                       # 0,1 % du volume de transferts

capex   = 80      # investissement initial (année 0), millions de DH
taux_is = 0.40    # impôt sur les sociétés des établissements de crédit

print(f"📈 Rendement moyen pondéré des réserves : {rendement_reserves * 100:.3f} %")

📈 Rendement moyen pondéré des réserves : 1.794 %


**Résultat :** compte tenu de l'allocation retenue (20 % de liquidités non rémunérées),
le rendement moyen pondéré des réserves s'établit à **1,794 %**, soit un niveau
inférieur aux taux des bons du Trésor : c'est le coût de la sécurité de liquidité.

In [25]:
pl = pd.DataFrame(index=annees)

# Volumes
pl["Encours (Mds DH)"]    = encours
pl["Volume MRE (Mds DH)"] = [transferts_mre * p for p in part_mre]

# Revenus (millions de DH) : × 1000 pour passer des milliards aux millions
pl["Revenus des réserves"]     = pl["Encours (Mds DH)"] * 1000 * rendement_reserves
pl["Commissions de transfert"] = pl["Volume MRE (Mds DH)"] * 1000 * commission
pl["Chiffre d'affaires"]       = pl["Revenus des réserves"] + pl["Commissions de transfert"]

# Charges (millions de DH)
pl["Coûts fixes"]     = cout_tech + cout_conformite + cout_personnel + np.array(cout_marketing)
pl["Coûts variables"] = pl["Volume MRE (Mds DH)"] * 1000 * cout_variable
pl["Total charges"]   = pl["Coûts fixes"] + pl["Coûts variables"]

# Résultat
pl["Résultat avant impôt"] = pl["Chiffre d'affaires"] - pl["Total charges"]
pl["Impôt (IS 40 %)"]      = pl["Résultat avant impôt"].clip(lower=0) * taux_is
pl["Résultat net"]         = pl["Résultat avant impôt"] - pl["Impôt (IS 40 %)"]

pl.index = [f"Année {a}" for a in annees]
pl.T.round(1)

,Année 1,Année 2,Année 3,Année 4,Année 5
Encours (Mds DH),1.0,3.0,6.0,9.0,12.0
Volume MRE (Mds DH),2.4,6.1,12.2,18.3,24.4
Revenus des réserves,17.9,53.8,107.6,161.5,215.3
Commissions de transfert,12.2,30.5,61.0,91.5,122.0
Chiffre d'affaires,30.1,84.3,168.6,253.0,337.3
Coûts fixes,90.0,90.0,85.0,85.0,85.0
Coûts variables,2.4,6.1,12.2,18.3,24.4
Total charges,92.4,96.1,97.2,103.3,109.4
Résultat avant impôt,-62.3,-11.8,71.4,149.7,227.9
Impôt (IS 40 %),0.0,0.0,28.6,59.9,91.1


**Interprétation du compte de résultat :**

- **Une phase d'investissement de deux ans.** Le MADc enregistre des pertes en année 1
  (−62,3 M DH) et en année 2 (−11,8 M DH), les coûts fixes (90 M DH) n'étant pas encore
  couverts par un encours trop faible.
- **Un seuil de rentabilité atteint en année 3**, avec un résultat net de 42,9 M DH,
  qui progresse ensuite jusqu'à **136,7 M DH en année 5**, soit une marge nette d'environ 40 %.
- **Un fort levier opérationnel.** Les charges ne progressent que de 92 à 109 M DH
  sur la période, alors que le chiffre d'affaires est multiplié par 11 (de 30 à 337 M DH).
  Les coûts étant majoritairement fixes, la croissance de l'encours se transforme
  presque intégralement en résultat.
- **Une forte dépendance aux taux d'intérêt.** En année 5, les revenus des réserves
  représentent **64 %** du chiffre d'affaires (215,3 M DH), contre 36 % pour les commissions
  de transfert. La rentabilité du MADc est donc directement exposée aux décisions
  de politique monétaire de Bank Al-Maghrib.

**Limites :**
- le modèle ne tient pas compte du **report des déficits** autorisé par la fiscalité
  marocaine, qui réduirait l'impôt des années 3 et 4 ;
- les revenus sont calculés sur l'encours de **fin d'année**, ce qui surestime
  légèrement les revenus en phase de croissance.

In [26]:
fig = px.bar(pl, x=pl.index, y=["Chiffre d'affaires", "Total charges"], barmode="group",
             title="MADc : chiffre d'affaires, charges et résultat net (millions de DH)")

fig.add_scatter(x=pl.index, y=pl["Résultat net"], mode="lines+markers+text",
                name="Résultat net", text=pl["Résultat net"].round(1),
                textposition="top center")

fig.add_hline(y=0, line_color="black")     # ligne du zéro : au-dessous, c'est une perte
fig.update_layout(yaxis_title="Millions de DH", xaxis_title="", legend_title="")
fig.show()

**Interprétation du graphique :**
Le croisement des courbes illustre le modèle économique du MADc : les charges forment
un socle quasi constant, tandis que le chiffre d'affaires croît avec l'encours et
l'adoption. Le résultat net devient positif entre la deuxième et la troisième année,
puis progresse de manière quasi linéaire. La rentabilité dépend donc avant tout de
la capacité à **attirer rapidement un encours suffisant** pour absorber les coûts fixes.

### 7.2 Évaluation financière : VAN, TRI et délai de récupération

Les flux de trésorerie du projet sont constitués de l'investissement initial (−80 M DH
en année 0) et des résultats nets des années 1 à 5. Ils sont actualisés au taux de
**10 %**, correspondant au rendement minimum exigé pour un projet innovant et risqué.

- **VAN** (valeur actuelle nette) : création de valeur du projet en dirhams d'aujourd'hui ;
- **TRI** (taux de rentabilité interne) : rendement annuel implicite du projet ;
- **Délai de récupération** : temps nécessaire pour récupérer l'investissement initial.

L'horizon est volontairement limité à 5 ans, sans valeur terminale, ce qui constitue
une approche **prudente**.

In [27]:
!pip install -q numpy-financial
import numpy_financial as npf

taux_actualisation = 0.10    # rendement minimum exigé

# Flux : année 0 = investissement, années 1 à 5 = résultat net
flux = [-capex] + list(pl["Résultat net"])

van = npf.npv(taux_actualisation, flux)
tri = npf.irr(flux)

print(f"💰 VAN (à 10 %) : {van:.1f} millions de DH")
print(f"📊 TRI : {tri * 100:.1f} %\n")

tableau_flux = pd.DataFrame({
    "Flux de trésorerie (M DH)": flux,
    "Flux cumulé (M DH)": np.cumsum(flux),
}, index=["Année 0"] + list(pl.index)).round(1)

tableau_flux

💰 VAN (à 10 %) : 32.0 millions de DH
📊 TRI : 15.9 %



,Flux de trésorerie (M DH),Flux cumulé (M DH)
Année 0,-80.0,-80.0
Année 1,-62.3,-142.3
Année 2,-11.8,-154.1
Année 3,42.9,-111.2
Année 4,89.8,-21.4
Année 5,136.7,115.3


**Interprétation :**

- **Le projet crée de la valeur** : la VAN au taux de 10 % s'élève à **+32,0 M DH**
  et le TRI atteint **15,9 %**, supérieur au rendement minimum exigé de 10 %.
- **Un délai de récupération d'environ 4 ans et 2 mois.** Le flux cumulé ne devient positif
  qu'au cours de l'année 5 (−21,4 M DH à fin d'année 4, puis +115,3 M DH à fin d'année 5).
- **Un besoin de financement maximal de 154 M DH** atteint à la fin de l'année 2 :
  l'investissement initial (80 M DH) s'ajoute aux pertes des deux premières années.
  La banque émettrice doit donc être en mesure de supporter ce creux de trésorerie.

Ces résultats sont obtenus sur un horizon de 5 ans **sans valeur terminale** : la valeur
réelle du projet, qui continuerait de générer des profits au-delà, est donc sous-estimée.

### 7.3 Sensibilité aux taux d'intérêt

Les revenus du MADc reposant majoritairement sur le rendement des réserves, nous mesurons
l'impact d'une variation des taux des bons du Trésor de −100 à +100 pb
sur la VAN du projet, toutes choses égales par ailleurs.

In [28]:
def van_selon_taux(choc_pb):
    """Recalcule la VAN si les taux des bons du Trésor varient de choc_pb points de base."""
    choc = choc_pb / 10000
    rendement = part_bt13 * (taux_bt13 + choc) + part_bt52 * (taux_bt52 + choc)  # le cash reste à 0 %

    revenus  = pl["Encours (Mds DH)"] * 1000 * rendement + pl["Commissions de transfert"]
    rai      = revenus - pl["Total charges"]                 # résultat avant impôt
    resultat = rai - rai.clip(lower=0) * taux_is             # résultat net

    return npf.npv(taux_actualisation, [-capex] + list(resultat))

chocs = [-100, -50, 0, 50, 100]

sens_taux = pd.DataFrame({
    "Choc de taux (pb)": chocs,
    "VAN (M DH)": [round(van_selon_taux(c), 1) for c in chocs],
})

fig = px.bar(sens_taux, x="Choc de taux (pb)", y="VAN (M DH)", text="VAN (M DH)",
             title="Sensibilité de la VAN du MADc aux taux des bons du Trésor",
             color=sens_taux["VAN (M DH)"] > 0,
             color_discrete_map={True: "seagreen", False: "indianred"})

fig.add_hline(y=0, line_color="black")
fig.update_traces(textposition="outside")
fig.update_xaxes(type="category")
fig.update_layout(showlegend=False)
fig.show()

sens_taux

,Choc de taux (pb),VAN (M DH)
0,-100,-82.0
1,-50,-25.0
2,0,32.0
3,50,89.0
4,100,142.0


**Interprétation de la sensibilité aux taux :**

| Choc sur les taux des bons du Trésor | VAN |
|---|---|
| −100 pb | −82,0 M DH |
| −50 pb | −25,0 M DH |
| 0 (scénario central) | +32,0 M DH |
| +50 pb | +89,0 M DH |
| +100 pb | +142,0 M DH |

- **La VAN est très sensible aux taux :** chaque variation de 50 pb la modifie
  d'environ **57 M DH**, soit près de deux fois la VAN du scénario central.
- **Le seuil critique est proche :** une baisse d'environ **30 pb** des taux suffirait
  à annuler la création de valeur. Le projet n'est donc pas viable dans un environnement
  de taux en baisse.
- **Cette fragilité découle du modèle économique :** les revenus des réserves
  représentent la majorité du chiffre d'affaires. Le MADc est, de fait, un **pari
  sur le niveau des taux** fixés par la politique monétaire de Bank Al-Maghrib.

**Leviers pour réduire cette dépendance :**
- **diversifier les revenus** : paiements marchands, services aux entreprises,
  hausse des volumes de transferts ;
- **réduire les coûts fixes** grâce à des infrastructures mutualisées ;
- **allonger la durée des placements** pour garantir des taux plus longtemps,
  au prix d'un **risque de liquidité** plus élevé en cas de retraits massifs,
  exactement le mécanisme qui a conduit à la faillite de SVB.

### Conclusion de la partie 7

Dans le scénario central, le MADc est un projet **rentable** : seuil de rentabilité
atteint dès l'année 3, résultat net de 136,7 M DH en année 5, VAN de +32 M DH et
TRI de 15,9 %. Son modèle repose sur un fort levier opérationnel : une fois les coûts
fixes couverts, la croissance de l'encours se transforme presque entièrement en profit.

Cette rentabilité reste toutefois **fragile** : elle dépend majoritairement des intérêts
perçus sur les réserves, et une baisse des taux d'environ 30 pb suffirait à annuler
la création de valeur.

L'émetteur fait face à un **arbitrage central** : investir davantage les réserves
améliore la rentabilité, mais réduit la liquidité disponible en cas de retraits massifs.
La partie suivante teste précisément cette résilience au moyen d'une simulation
de Monte Carlo.

## 8. Stress test de liquidité : simulation de Monte Carlo

Nous testons la capacité du MADc à faire face à une **ruée sur les rachats** (*bank run*)
lorsque l'encours atteint 12 milliards de DH (année 5).

**Principe :** 10 000 crises de 10 jours sont simulées. Pour chacune, l'intensité
(part de l'encours retirée) est tirée aléatoirement selon une loi Bêta (moyenne ≈ 11 %,
avec des scénarios extrêmes rares). Les retraits sont concentrés sur les premiers jours,
à l'image des paniques bancaires observées (SVB, 2023).

**Mobilisation des réserves, dans l'ordre :**

| Réserve | Capacité de mobilisation | Coût de revente |
|---|---|---|
| Liquidités | immédiate | aucun |
| BT 13 semaines | 3 % de l'encours par jour | 0,25 % |
| BT 52 semaines | 1 % de l'encours par jour | 2 % (vente forcée) |

Une **rupture de liquidité** survient lorsque les retraits d'une journée dépassent
les ressources mobilisables : l'émetteur ne peut plus honorer les rachats à temps,
ce qui menace le peg.

In [30]:
def simuler_crises(part_cash, n_simulations=10000, seed=42):
    """Simule n crises de 10 jours et indique, pour chacune, s'il y a rupture de liquidité."""
    rng = np.random.default_rng(seed)            # générateur de hasard reproductible

    E = 12_000                                   # encours : 12 milliards = 12 000 millions de DH
    poids = np.array([0.25, 0.20, 0.15, 0.10, 0.08, 0.07, 0.05, 0.04, 0.03, 0.03])  # panique au début
    part_placee = 1 - part_cash                  # le reste est placé en bons du Trésor (5/8 à 13 sem., 3/8 à 52 sem.)

    resultats = []

    for i in range(n_simulations):
        intensite = rng.beta(1.5, 12)            # % de l'encours retiré pendant la crise (tiré au hasard)
        retraits = intensite * E * poids         # retraits jour par jour

        # Les 3 réservoirs au début de la crise
        cash = part_cash * E
        bt13 = part_placee * 5 / 8 * E
        bt52 = part_placee * 3 / 8 * E

        pertes = 0
        rupture = False

        for besoin in retraits:                  # on avance jour par jour
            # ① payer avec le cash
            paye = min(cash, besoin)
            cash -= paye
            besoin -= paye

            # ② vendre des BT 13 semaines (3 % de l'encours max par jour, perte de 0,25 %)
            vente = min(bt13, 0.03 * E, besoin)
            bt13 -= vente
            besoin -= vente
            pertes += vente * 0.0025

            # ③ vendre des BT 52 semaines (1 % de l'encours max par jour, perte de 2 %)
            vente = min(bt52, 0.01 * E, besoin)
            bt52 -= vente
            besoin -= vente
            pertes += vente * 0.02

            # s'il reste des clients non payés : rupture
            if besoin > 0:
                rupture = True

        resultats.append({
            "Intensité (%)": intensite * 100,
            "Rupture": rupture,
            "Pertes de vente forcée (M DH)": pertes,
        })

    return pd.DataFrame(resultats)

In [31]:
sim = simuler_crises(part_cash=0.20)

proba_rupture = sim["Rupture"].mean() * 100
pertes_moy    = sim["Pertes de vente forcée (M DH)"].mean()
pertes_max    = sim["Pertes de vente forcée (M DH)"].max()
intensite_max = sim["Intensité (%)"].max()

print(f"🎲 Nombre de crises simulées       : {len(sim)}")
print(f"📊 Intensité moyenne des retraits   : {sim['Intensité (%)'].mean():.1f} % de l'encours")
print(f"🔥 Crise la plus grave simulée      : {intensite_max:.1f} % de l'encours")
print(f"🚨 Probabilité de rupture           : {proba_rupture:.2f} %")
print(f"💸 Pertes moyennes de vente forcée  : {pertes_moy:.2f} M DH")
print(f"💥 Pertes maximales de vente forcée : {pertes_max:.2f} M DH")

🎲 Nombre de crises simulées       : 10000
📊 Intensité moyenne des retraits   : 11.2 % de l'encours
🔥 Crise la plus grave simulée      : 58.9 % de l'encours
🚨 Probabilité de rupture           : 0.69 %
💸 Pertes moyennes de vente forcée  : 0.35 M DH
💥 Pertes maximales de vente forcée : 19.15 M DH


**Interprétation des résultats (20 % de liquidités) :**

- Sur 10 000 crises simulées, l'intensité moyenne des retraits atteint **11,2 %** de l'encours,
  et le scénario le plus sévère **58,9 %**, soit une panique plus forte que celle de SVB.
- **La probabilité de rupture de liquidité est de 0,69 %** : l'émetteur honore
  l'intégralité des rachats dans plus de 99 % des crises simulées.
- **Les pertes liées aux ventes forcées restent limitées** : 0,35 M DH en moyenne et
  19,15 M DH dans le pire scénario, soit environ 14 % du résultat net de l'année 5
  (136,7 M DH). L'émetteur n'est donc **jamais menacé de faillite**.

**Lecture :** le risque principal du MADc n'est pas un risque de **solvabilité** (manquer
d'argent) mais un risque de **liquidité** (ne pas pouvoir mobiliser l'argent assez vite).

*Note : la probabilité de 0,69 % est conditionnelle à la survenance d'une crise ;
elle ne correspond pas à une probabilité annuelle.*

In [32]:
sim["Rupture de liquidité"] = sim["Rupture"].map({True: "Oui", False: "Non"})

fig = px.histogram(sim, x="Intensité (%)", color="Rupture de liquidité", nbins=60,
                   color_discrete_map={"Non": "seagreen", "Oui": "indianred"},
                   title="10 000 crises simulées : intensité des retraits (20 % de liquidités)")

fig.add_vline(x=sim["Intensité (%)"].mean(), line_dash="dash", line_color="black",
              annotation_text="Moyenne", annotation_position="top right")

fig.update_layout(yaxis_title="Nombre de crises", xaxis_title="Part de l'encours retirée (%)")
fig.show()

**Interprétation de l'histogramme :**
- La distribution est **fortement asymétrique** : la grande majorité des crises
  entraîne des retraits compris entre 2 % et 15 % de l'encours, absorbés sans difficulté.
- Une **queue de distribution** s'étend vers les crises extrêmes, rares mais possibles.
- **Les ruptures apparaissent uniquement au-delà d'environ 40 % de retraits** en 10 jours.
  En deçà, la combinaison des liquidités (20 %) et des cessions progressives de bons
  du Trésor suffit à honorer tous les rachats.

Le seuil de résistance du MADc est donc d'environ **40 % de l'encours retiré en 10 jours**.
La question devient alors : **ce niveau de 20 % de liquidités est-il optimal ?**

### 8.2 Arbitrage entre sécurité et rentabilité

Conserver davantage de liquidités réduit le risque de rupture, mais diminue les revenus
issus du placement des réserves (**coût d'opportunité** d'environ 2,24 % par an).
Nous testons donc plusieurs niveaux de liquidités, de 5 % à 40 % de l'encours,
face aux **mêmes 10 000 crises**, afin d'identifier le niveau optimal.

Le **seuil de tolérance** retenu est une probabilité de rupture de **1 %**.

In [33]:
niveaux_cash = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40]

taux_place = 5/8 * taux_bt13 + 3/8 * taux_bt52    # rendement des réserves placées (≈ 2,24 %)
E = 12_000                                       # encours de l'année 5 en millions de DH

lignes = []
for c in niveaux_cash:
    s = simuler_crises(part_cash=c)              # les mêmes 10 000 crises (seed=42)
    lignes.append({
        "Part de cash (%)": int(c * 100),
        "Probabilité de rupture (%)": round(s["Rupture"].mean() * 100, 2),
        "Pertes moyennes (M DH)": round(s["Pertes de vente forcée (M DH)"].mean(), 2),
        "Revenus des réserves an 5 (M DH)": round(E * (1 - c) * taux_place, 1),
    })

arbitrage = pd.DataFrame(lignes)
arbitrage

,Part de cash (%),Probabilité de rupture (%),Pertes moyennes (M DH),Revenus des réserves an 5 (M DH)
0,5,14.38,2.58,255.6
1,10,5.90,1.41,242.2
2,15,2.55,0.70,228.7
3,20,0.69,0.35,215.3
4,25,0.56,0.17,201.8
5,30,0.13,0.07,188.4
6,40,0.01,0.01,161.5


**Interprétation du tableau :**

- **Le niveau de liquidités détermine fortement la résistance du MADc.** La probabilité
  de rupture passe de **14,38 %** avec 5 % de liquidités à **0,69 %** avec 20 %,
  puis devient quasi nulle à 40 % (0,01 %).
- **La sécurité a un coût constant :** chaque tranche supplémentaire de 5 % de liquidités
  réduit les revenus des réserves d'environ **13,5 M DH par an** (année 5).
- **Les gains de sécurité sont décroissants.** Passer de 5 % à 10 % réduit le risque
  de 8,5 points, tandis que passer de 20 % à 25 % ne le réduit que de 0,13 point,
  pour un coût identique.
- Les pertes liées aux ventes forcées restent faibles dans tous les cas (2,58 M DH
  en moyenne au maximum), ce qui confirme que le risque est un risque de **liquidité**
  et non de **solvabilité**.

In [34]:
fig = px.line(arbitrage, x="Revenus des réserves an 5 (M DH)", y="Probabilité de rupture (%)",
              text="Part de cash (%)", markers=True,
              title="Arbitrage sécurité / rentabilité selon la part de liquidités")

fig.update_traces(texttemplate="%{text} % cash", textposition="top right")
fig.add_hline(y=1, line_dash="dash", line_color="red",
              annotation_text="Seuil de tolérance : 1 %", annotation_position="top left")
fig.show()

**Interprétation du graphique : la frontière sécurité / rentabilité**

La courbe représente l'ensemble des compromis possibles entre revenus (axe horizontal)
et risque de rupture (axe vertical). Elle présente un **coude** marqué autour de
**20 % de liquidités** :
- à droite du coude, une faible hausse des revenus s'accompagne d'une forte hausse du risque ;
- à gauche du coude, une forte baisse des revenus ne procure presque plus de gain de sécurité.

**Avec un seuil de tolérance de 1 %, le niveau optimal est de 20 % de liquidités** :
c'est l'allocation la plus rentable (215,3 M DH de revenus) qui respecte la contrainte
de risque. Un émetteur plus prudent, visant un risque inférieur à 0,25 %, devrait
porter cette part à 30 %, au prix d'environ 27 M DH de revenus annuels en moins.

### Conclusion de la partie 8

Le stress test de Monte Carlo montre qu'un stablecoin dirham conservant **20 % de ses
réserves en liquidités** résiste à plus de 99 % des crises simulées, y compris à des
retraits représentant jusqu'à **40 % de l'encours en 10 jours**, soit une panique
d'une ampleur comparable à celle de SVB.

Trois enseignements se dégagent :
1. **Le risque d'un stablecoin adossé à des réserves de qualité est un risque de liquidité,
   et non de solvabilité** : même dans le pire scénario, les pertes restent limitées
   (19 M DH), mais le rythme de mobilisation des réserves peut être insuffisant.
2. **Il existe un niveau optimal de liquidités**, autour de 20 %, au-delà duquel le coût
   de la sécurité devient disproportionné.
3. **Sécurité et rentabilité sont étroitement liées.** Or la partie 7 a montré que la VAN
   devient négative si les taux baissent d'environ 30 pb. La marge de manœuvre de
   l'émetteur est donc étroite : il ne peut ni réduire sa liquidité sans danger,
   ni l'augmenter sans compromettre sa rentabilité.

**Recommandation :** le cadre réglementaire devrait imposer un **ratio minimum de
liquidités** (de l'ordre de 20 % de l'encours) et prévoir un **accès à la liquidité de
Bank Al-Maghrib** en cas de crise, adossé aux bons du Trésor détenus. Ce mécanisme
réduirait fortement le risque de rupture sans pénaliser la rentabilité.

In [35]:
bas.to_csv("prix_bas_stablecoins.csv")
print("💾 Fichier sauvegardé")

💾 Fichier sauvegardé


In [36]:
import yfinance as yf
import pandas as pd
from google.colab import files

tickers = {"USDT": "USDT-USD", "USDC": "USDC-USD", "DAI": "DAI-USD"}

series_close, series_low = [], []
for nom, code in tickers.items():
    historique = yf.Ticker(code).history(start="2020-01-01")
    series_close.append(historique["Close"].rename(nom))   # prix de clôture
    series_low.append(historique["Low"].rename(nom))       # prix le plus bas du jour

prix_export = pd.concat(series_close, axis=1)
bas_export  = pd.concat(series_low, axis=1)
prix_export.index = prix_export.index.tz_localize(None)
bas_export.index  = bas_export.index.tz_localize(None)

# Créer les 2 fichiers
prix_export.to_csv("prix_stablecoins.csv")
bas_export.to_csv("prix_bas_stablecoins.csv")
print("💾 Fichiers créés :", prix_export.shape, bas_export.shape)

# Les télécharger sur ton PC
files.download("prix_stablecoins.csv")
files.download("prix_bas_stablecoins.csv")

💾 Fichiers créés : (2494, 3) (2494, 3)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>